# Seoul Bike-Sharing Demand and Service Planning

## A basic-to-intermediate Software Systems & Services Management project

**Purpose:** show how simple data analysis can support a clear service-management decision.  
**Decision-maker:** citywide bike-sharing operations manager.  
**Main decision:** when should the service use normal, watch, high, or critical readiness?  
**Analytical levels:** descriptive analysis, basic multiple linear regression for association, and transparent service-planning rules.  
**Not used:** forecasting, train/test evaluation, accuracy metrics, causal claims, station optimisation, or financial forecasting.

### Central business question

> How can historical hourly demand, weather, and calendar patterns help a bike-sharing operator plan readiness and communicate responsibly with riders?

The complete project follows this chain:

> **What happened? → What remains associated? → What should management do?**

**Source:** UCI Machine Learning Repository, Seoul Bike Sharing Demand  
**DOI:** https://doi.org/10.24432/C5F62R  
**Coverage:** 1 December 2017 to 30 November 2018; 8,760 citywide hourly records.

## 1. Why this project fits Software Systems & Services Management

The project is not mainly about advanced statistics. It treats bike sharing as a service system involving data, users, operations, communication, and management rules.

| Programme idea | Evidence in this project |
|---|---|
| Service science | Connects rider needs with operational decisions |
| Systems thinking | Explains feedback between demand, readiness, availability, and trust |
| Domain understanding | Defines stakeholders, data entities, decisions, and boundaries |
| Business process management | Maps how an hourly advisory becomes an operational response |
| Smart service design | Uses weather and calendar information in a transparent decision aid |
| Information-systems management | Defines owners, KPIs, review steps, and data limitations |

### Research questions

1. Is the dataset reliable enough for an educational service prototype?
2. At which hours and weekdays is demand highest?
3. How does demand differ by season, rain, calendar type, and temperature band?
4. When time, calendar, season, and weather are considered together, which factors remain associated with rentals?
5. How can the combined evidence become simple readiness rules and service actions?
6. What can this dataset not tell management?

## 2. Stakeholders and scope

| Stakeholder | Need | Project output |
|---|---|---|
| Rider | Clear information without false guarantees | Citywide demand advisory |
| Operations manager | Know when extra attention may be needed | Readiness level and checklist |
| Field and maintenance team | Avoid preventable conflict with busy periods | Suggested low-pressure work windows |
| City transport planner | Understand seasonal mobility patterns | Monthly and seasonal evidence |
| Data/service team | Keep rules understandable and reviewable | Lookup table, KPI definitions, limitations |

**Scope boundary:** one row is one citywide hour. The data contain no station, bicycle inventory, route, staffing, cost, customer, or failed-rental information. Therefore, the project cannot promise station availability or calculate exact resource requirements.

## 3. Setup

### What this code does

It imports the basic tools needed to load a table, calculate grouped summaries, estimate one regression equation, and create charts. The regression uses NumPy directly so every step remains visible.

In [ ]:
# Path, ZIP, and download tools help the notebook find the official CSV.
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen
import warnings

# pandas works with tables; numpy supports a few numerical operations.
import pandas as pd
import numpy as np

# matplotlib and seaborn create clear presentation charts.
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings make tables and charts easier to read.
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.1f}")
sns.set_theme(style="whitegrid", context="notebook")

COLORS = {"navy": "#173F5F", "blue": "#20639B", "teal": "#3CAEA3",
          "gold": "#F6D55C", "red": "#ED553B", "grey": "#6B7280"}

## 4. Load the official data

### Business question

**Can another person reproduce the project from a documented source?**

The code first looks for the included CSV. If it is not present, it downloads the official UCI ZIP. The unusual `cp949` encoding belongs to the original file.

In [ ]:
# Check simple local paths first so the notebook can work offline.
local_options = [Path("data/seoul_bike/SeoulBikeData.csv"), Path("SeoulBikeData.csv")]
local_path = next((path for path in local_options if path.exists()), None)

if local_path is not None:
    raw = pd.read_csv(local_path, encoding="cp949")
    source_used = str(local_path)
else:
    # Download only when the local file is missing.
    url = "https://archive.ics.uci.edu/static/public/560/seoul%2Bbike%2Bsharing%2Bdemand.zip"
    with urlopen(url) as response:
        downloaded_zip = BytesIO(response.read())
    with ZipFile(downloaded_zip) as archive:
        with archive.open("SeoulBikeData.csv") as csv_file:
            raw = pd.read_csv(csv_file, encoding="cp949")
    source_used = "Official UCI ZIP"

print(f"Source used: {source_used}")
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
display(raw.head(3))

## 5. Prepare understandable variables

### Business question

**How can the original columns be made easier to analyse and explain?**

The code renames fields, converts the date, and adds weekday, month, day type, weather flags, temperature bands, and operating periods. These are simple labels derived directly from the source values.

In [ ]:
# Keep the imported table unchanged by working on a copy.
df = raw.copy()

# Short names reduce typing errors and are easier to explain.
df.columns = [
    "date", "rented_bikes", "hour", "temperature_c", "humidity_pct",
    "wind_speed_ms", "visibility_10m", "dew_point_c", "solar_radiation_mjm2",
    "rainfall_mm", "snowfall_cm", "season", "holiday", "functioning_day"
]

# dayfirst=True is necessary because the source date is day/month/year.
df["date"] = pd.to_datetime(df["date"], dayfirst=True)
df["weekday"] = df["date"].dt.day_name()
df["day_number"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.month_name().str[:3]
df["day_type"] = np.where(df["day_number"] >= 5, "Weekend", "Weekday")
df["calendar_type"] = np.select(
    [df["holiday"].eq("Holiday"), df["day_number"].ge(5)],
    ["Holiday", "Weekend"],
    default="Working day"
)
df["rain_condition"] = np.where(df["rainfall_mm"] > 0, "Recorded rain", "No recorded rain")
df["rain_present"] = (df["rainfall_mm"] > 0).astype(int)
df["snow_condition"] = np.where(df["snowfall_cm"] > 0, "Recorded snow", "No recorded snow")

# Broad bands make temperature comparisons easier to present than many exact values.
df["temperature_band"] = pd.cut(
    df["temperature_c"],
    bins=[-30, 0, 10, 20, 30, 45],
    labels=["Below 0°C", "0–10°C", "10–20°C", "20–30°C", "Above 30°C"]
)

# Named periods translate 24 separate hours into operational language.
df["operating_period"] = pd.cut(
    df["hour"], bins=[-1, 5, 9, 15, 19, 23],
    labels=["Overnight", "Morning commute", "Midday", "Evening commute", "Late evening"]
)

display(df[["date", "hour", "rented_bikes", "weekday", "day_type", "season",
            "rain_condition", "temperature_band", "operating_period"]].head())

## Analytical Level 1 — Descriptive analysis: What happened?

Descriptive analysis summarises the observed year. It identifies recurring time, calendar, seasonal, and weather patterns without trying to predict a future hour.

### 6. RQ1 — Data-quality audit

### Business question

**Is the data complete and logically consistent enough for descriptive analysis?**

The code counts missing and duplicate rows, checks the date range, and tests whether each date has 24 hourly records. It also separates non-functioning hours because zero rentals during a shutdown do not represent customer demand.

In [ ]:
# Basic checks answer different quality questions.
quality_summary = pd.DataFrame({
    "Check": ["Rows", "Missing cells", "Duplicate rows", "Unique dates",
              "Dates with exactly 24 rows", "Non-functioning hours"],
    "Result": [len(df), int(df.isna().sum().sum()), int(df.duplicated().sum()),
               df["date"].nunique(), int((df.groupby("date").size() == 24).sum()),
               int((df["functioning_day"] == "No").sum())]
})
display(quality_summary)

# Demand analysis uses functioning hours only; shutdown zeros are a different event.
operating = df[df["functioning_day"] == "Yes"].copy()

print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Functioning hours used for demand analysis: {len(operating):,}")
print(f"Average rentals per functioning hour: {operating['rented_bikes'].mean():,.1f}")

### Finding and business meaning

The table is suitable for a one-year citywide educational case, but non-functioning hours must be excluded from demand comparisons. This is a service-management choice: a shutdown is an availability failure, not evidence of zero customer interest.

## 7. RQ2 — When does demand create the most pressure?

### Business question

**Which combinations of hour and weekday should operations monitor most closely?**

The code groups functioning records by weekday and hour, calculates the mean, and displays a heatmap. This is simply an average of comparable historical hours.

In [ ]:
# Fix the display order so the chart reads Monday through Sunday.
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Mean rentals for each weekday-hour combination create a historical timetable view.
hour_weekday = operating.pivot_table(
    index="weekday", columns="hour", values="rented_bikes", aggfunc="mean"
).reindex(weekday_order)

plt.figure(figsize=(14, 5))
sns.heatmap(hour_weekday, cmap="YlGnBu", cbar_kws={"label": "Average rentals per hour"})
plt.title("Commute hours create the clearest recurring demand pressure", weight="bold")
plt.xlabel("Hour of day")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Sort the table so the strongest historical combinations appear first.
top_time_windows = (
    operating.groupby(["weekday", "hour"], as_index=False)["rented_bikes"].mean()
    .rename(columns={"rented_bikes": "average_rentals"})
    .sort_values("average_rentals", ascending=False)
    .head(10)
)
display(top_time_windows)

### How to present the conclusion

> “The heatmap shows recurring time pressure, especially around commuting hours. Management can therefore schedule monitoring and readiness checks before these windows rather than reacting after demand rises.”

This does **not** prove that commuting causes the demand pattern; it only identifies a useful repeated association.

## 8. RQ3 — How does season change the operating context?

### Business question

**Should the same readiness plan be used throughout the year?**

The code compares mean hourly rentals by season and also shows the monthly pattern. Both are descriptive summaries.

In [ ]:
season_order = ["Winter", "Spring", "Summer", "Autumn"]
season_summary = (
    operating.groupby("season")["rented_bikes"]
    .agg(average_rentals="mean", total_rentals="sum", observed_hours="size")
    .reindex(season_order)
    .reset_index()
)

month_order = list(range(1, 13))
monthly = operating.groupby("month")["rented_bikes"].mean().reindex(month_order)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=season_summary, x="season", y="average_rentals", color=COLORS["teal"], ax=axes[0])
axes[0].set_title("Seasonal demand makes one fixed readiness plan unsuitable", weight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("Average rentals per functioning hour")

axes[1].plot(monthly.index, monthly.values, marker="o", color=COLORS["blue"], linewidth=2.5)
axes[1].set_title("Monthly averages show how pressure changes through the year", weight="bold")
axes[1].set_xlabel("Month number")
axes[1].set_ylabel("Average rentals per functioning hour")
axes[1].set_xticks(month_order)
plt.tight_layout()
plt.show()

display(season_summary)

### Business meaning

Seasonal planning should be reviewed before each season. The comparison is useful for scheduling preparedness, but it cannot calculate the exact number of bicycles or employees needed because the required capacity and cost data are absent.

## 9. RQ3 — How do recorded rain and temperature relate to demand?

### Business question

**Should weather information modify the normal timetable plan?**

The code compares averages for rainy and non-rainy hours, then compares broad temperature bands. The percentage difference is calculated as:

\[
\frac{\text{rain average} - \text{dry average}}{\text{dry average}} \times 100
\]

This describes an association in the observed data. It does not prove that weather alone caused the difference.

In [ ]:
rain_summary = (
    operating.groupby("rain_condition")["rented_bikes"]
    .agg(average_rentals="mean", observed_hours="size")
    .reset_index()
)

rain_avg = rain_summary.loc[rain_summary["rain_condition"] == "Recorded rain", "average_rentals"].iloc[0]
dry_avg = rain_summary.loc[rain_summary["rain_condition"] == "No recorded rain", "average_rentals"].iloc[0]
rain_difference_pct = (rain_avg - dry_avg) / dry_avg * 100

temperature_summary = (
    operating.groupby("temperature_band", observed=False)["rented_bikes"]
    .agg(average_rentals="mean", observed_hours="size")
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=rain_summary, x="rain_condition", y="average_rentals", color=COLORS["blue"], ax=axes[0])
axes[0].set_title("Recorded rain is associated with much lower demand", weight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("Average rentals per functioning hour")

sns.barplot(data=temperature_summary, x="temperature_band", y="average_rentals", color=COLORS["gold"], ax=axes[1])
axes[1].set_title("Demand differs clearly across temperature bands", weight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Average rentals per functioning hour")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

display(rain_summary)
display(temperature_summary)
print(f"Observed rainy-hour average was {abs(rain_difference_pct):.1f}% lower than the dry-hour average.")

### Business meaning

The normal time-based plan should be checked against current or forecast weather. The service should not interpret lower demand as “no operational work”: lower-pressure periods may also be useful for planned maintenance, subject to safety and staffing rules.

## 10. RQ3 — Do holidays change the pattern?

### Business question

**Can a standard weekday plan be copied to holidays?**

The code compares average demand by holiday status and operating period. This avoids relying on one overall holiday number that may hide differences across the day.

In [ ]:
holiday_period = (
    operating.groupby(["holiday", "operating_period"], observed=False)["rented_bikes"]
    .mean().reset_index(name="average_rentals")
)

plt.figure(figsize=(11, 4.8))
sns.barplot(data=holiday_period, x="operating_period", y="average_rentals", hue="holiday",
            palette=[COLORS["blue"], COLORS["gold"]])
plt.title("Holiday status changes the shape of demand across the day", weight="bold")
plt.xlabel("")
plt.ylabel("Average rentals per functioning hour")
plt.xticks(rotation=15)
plt.legend(title="")
plt.tight_layout()
plt.show()

display(holiday_period)

### Business meaning

Holiday and normal-day patterns should be reviewed separately. This is a calendar rule that an operations manager can understand and audit without knowing any modelling technique.

## Analytical Level 2 — Multiple regression: What remains associated?

### Business question

> After accounting for hour, season, calendar type, temperature, humidity, and recorded rain together, which observed conditions are associated with higher or lower hourly rentals?

This is a **basic multiple linear regression** because it uses several inputs. It is used only to estimate associations within the historical dataset.

### Why some inputs are categorical

- **Hour:** demand has morning and evening peaks, so it does not change in one straight line from 00:00 to 23:00.
- **Season:** winter, spring, summer, and autumn are groups—not numerical amounts.
- **Calendar type:** working days, weekends, and holidays are groups—not a numerical scale.

The regression converts each group into indicator columns. It compares every category with a reference: **00:00, winter, and working day**.

### Interpretation boundary

A coefficient is an estimated average difference while the other included inputs are held constant. It does **not** prove causation and is **not** used here to forecast future demand.

In [ ]:
# Create an analysis table containing only the variables used in the equation.
regression_data = operating[[
    "rented_bikes", "hour", "season", "calendar_type",
    "temperature_c", "humidity_pct", "rain_present"
]].copy()

# Rescale temperature and humidity so the coefficients are easier to explain.
# A coefficient now represents +5°C or +10 humidity percentage points.
regression_data["temperature_plus_5c"] = regression_data["temperature_c"] / 5
regression_data["humidity_plus_10pct"] = regression_data["humidity_pct"] / 10

# Set the reference categories explicitly.
regression_data["hour"] = pd.Categorical(
    regression_data["hour"], categories=list(range(24)), ordered=True
)
regression_data["season"] = pd.Categorical(
    regression_data["season"], categories=["Winter", "Spring", "Summer", "Autumn"], ordered=True
)
regression_data["calendar_type"] = pd.Categorical(
    regression_data["calendar_type"],
    categories=["Working day", "Weekend", "Holiday"], ordered=True
)

# Start the design table with the intercept and the three direct numerical inputs.
X = pd.DataFrame({
    "Intercept": 1.0,
    "Temperature: +5°C": regression_data["temperature_plus_5c"],
    "Humidity: +10 points": regression_data["humidity_plus_10pct"],
    "Rain recorded": regression_data["rain_present"]
})

# Convert categorical variables into 0/1 indicator columns.
# drop_first=True leaves one reference category out of each group.
category_columns = pd.get_dummies(
    regression_data[["hour", "season", "calendar_type"]],
    drop_first=True, dtype=float
)
X = pd.concat([X, category_columns], axis=1)

# Estimate the coefficients that minimise the squared differences between
# observed rentals and the values represented by the fitted equation.
y = regression_data["rented_bikes"].to_numpy(float)
coefficients = np.linalg.lstsq(X.to_numpy(float), y, rcond=None)[0]
coefficient_table = pd.Series(coefficients, index=X.columns, name="estimated_difference")

# Select a small, readable group for explanation. Hour effects are displayed separately.
selected_coefficients = pd.Series({
    "Temperature (+5°C)": coefficient_table["Temperature: +5°C"],
    "Humidity (+10 points)": coefficient_table["Humidity: +10 points"],
    "Rain recorded": coefficient_table["Rain recorded"],
    "Weekend vs working day": coefficient_table["calendar_type_Weekend"],
    "Holiday vs working day": coefficient_table["calendar_type_Holiday"],
    "Spring vs winter": coefficient_table["season_Spring"],
    "Summer vs winter": coefficient_table["season_Summer"],
    "Autumn vs winter": coefficient_table["season_Autumn"]
}).sort_values()

hour_coefficients = pd.Series(
    {hour: coefficient_table[f"hour_{hour}"] for hour in range(1, 24)},
    name="difference_vs_00"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
bar_colors = [COLORS["red"] if value < 0 else COLORS["teal"] for value in selected_coefficients]
selected_coefficients.plot.barh(color=bar_colors, ax=axes[0])
axes[0].axvline(0, color=COLORS["navy"], linewidth=1)
axes[0].set_title("Selected factors show positive and negative associations", weight="bold")
axes[0].set_xlabel("Estimated difference in hourly rentals")
axes[0].set_ylabel("")

hour_coefficients.plot(color=COLORS["blue"], marker="o", ax=axes[1])
axes[1].axhline(0, color=COLORS["navy"], linewidth=1)
axes[1].set_title("Hour categories preserve the daily demand shape", weight="bold")
axes[1].set_xlabel("Hour (comparison with 00:00)")
axes[1].set_ylabel("Estimated difference in hourly rentals")
axes[1].set_xticks(range(1, 24, 2))
plt.tight_layout()
plt.show()

display(selected_coefficients.rename("Estimated rental difference").round(1).to_frame())

### How to explain the regression in the presentation

> “Descriptive charts examine factors separately. I added one basic multiple linear regression to consider time, calendar, season, and selected weather conditions together. Hour was categorical because rental demand has separate peaks and does not follow one straight daily trend. The coefficients are historical associations, not causal effects or future forecasts.”

The coefficient signs and sizes support scenario planning. They do not tell management the exact number of bicycles, staff, or station movements required.

## Analytical Level 3 — Service planning: What should management do?

Regression improves understanding, but management still needs transparent actions. The next steps combine descriptive patterns and regression findings with a simple, auditable historical lookup.

### 11. RQ4 — Which historical contexts usually had high demand?

### Business question

**Which simple combination of conditions should receive management attention?**

The code builds a historical lookup table using season, day type, hour, and rain condition. For each combination it calculates:

- number of observed hours;
- average rentals;
- median rentals; and
- share of hours above the overall 75th percentile.

The percentile is only a transparent way to define “historically high” within this dataset.

In [ ]:
# The 75th percentile separates the highest quarter of functioning-hour observations.
high_demand_cutoff = operating["rented_bikes"].quantile(0.75)

# Each row receives a simple yes/no high-demand label for summarisation.
operating["historically_high"] = operating["rented_bikes"] >= high_demand_cutoff

# This lookup table is a grouped historical summary, not a regression model.
context_lookup = (
    operating.groupby(["season", "day_type", "hour", "rain_condition"])
    .agg(
        observed_hours=("rented_bikes", "size"),
        average_rentals=("rented_bikes", "mean"),
        median_rentals=("rented_bikes", "median"),
        high_demand_share=("historically_high", "mean")
    )
    .reset_index()
)
context_lookup["high_demand_share_pct"] = context_lookup["high_demand_share"] * 100

# Requiring at least five observations prevents a tiny group from looking overly important.
top_contexts = (
    context_lookup[context_lookup["observed_hours"] >= 5]
    .sort_values(["high_demand_share", "average_rentals"], ascending=False)
    .head(12)
)

print(f"Historically high-demand cutoff (75th percentile): {high_demand_cutoff:,.0f} rentals/hour")
display(top_contexts[["season", "day_type", "hour", "rain_condition", "observed_hours",
                      "average_rentals", "median_rentals", "high_demand_share_pct"]])

### How to explain this in an interview

> “I did not train a predictive model. I grouped similar past hours and used their average and high-demand frequency as a historical reference. I required at least five observations before ranking a combination. The result supports planning, but it is not a guarantee of future demand.”

### 12. RQ5 — Convert evidence into transparent readiness levels

### Business question

**How can the analysis produce a consistent management response?**

The service uses the historical average for a selected context and compares it with overall demand quartiles. Quartiles divide functioning hours into four equal parts. The resulting level is a policy label—not a prediction or automated command.

In [ ]:
# Quartiles create four understandable demand bands from the observed distribution.
q25, q50, q75 = operating["rented_bikes"].quantile([0.25, 0.50, 0.75])

def readiness_level(historical_average):
    # Translate a historical average into one of four transparent policy bands.
    if historical_average < q25:
        return "Normal"
    if historical_average < q50:
        return "Watch"
    if historical_average < q75:
        return "High"
    return "Critical"

context_lookup["readiness_level"] = context_lookup["average_rentals"].apply(readiness_level)

service_actions = pd.DataFrame({
    "Readiness level": ["Normal", "Watch", "High", "Critical"],
    "Meaning": ["Historically low pressure", "Below-median pressure", "Above-median pressure", "Top-quartile pressure"],
    "Operator action": [
        "Use normal monitoring; consider suitable maintenance work",
        "Review weather and live service information",
        "Confirm operational coverage before the hour",
        "Escalate to operations lead and monitor live availability closely"
    ],
    "Rider message": [
        "Normal citywide demand is expected based on similar past conditions.",
        "Demand may increase; check live station availability before travel.",
        "Higher citywide demand is possible; allow extra time and check alternatives.",
        "Historically busy conditions; this is not a guarantee of station availability."
    ]
})

thresholds = pd.DataFrame({"Boundary": ["25th percentile", "Median", "75th percentile"],
                           "Rentals per hour": [q25, q50, q75]})
display(thresholds)
display(service_actions)

## 13. Simple decision-aid example

### What this code does

An operations user selects a season, day type, hour, and rain condition. The code looks for the matching row in the historical table and returns its average, sample size, readiness level, and written action. The operational rule uses the transparent lookup—not regression coefficients—so a manager can audit it directly.

In [ ]:
def historical_service_advice(season, day_type, hour, rain_condition):
    # Filter the lookup table to the user's selected conditions.
    match = context_lookup[
        (context_lookup["season"] == season) &
        (context_lookup["day_type"] == day_type) &
        (context_lookup["hour"] == hour) &
        (context_lookup["rain_condition"] == rain_condition)
    ]

    # A missing or very small group should trigger caution rather than a confident answer.
    if match.empty or int(match.iloc[0]["observed_hours"]) < 5:
        return pd.DataFrame({"Output": ["Result"], "Value": ["Insufficient similar historical observations"]})

    row = match.iloc[0]
    level = row["readiness_level"]
    action = service_actions.set_index("Readiness level").loc[level]

    return pd.DataFrame({
        "Output": ["Historical reference", "Similar hours", "Readiness level", "Operator action", "Rider message"],
        "Value": [
            f"{row['average_rentals']:,.0f} average citywide rentals/hour",
            f"{int(row['observed_hours'])} observations",
            level,
            action["Operator action"],
            action["Rider message"]
        ]
    })

# Example: a dry autumn weekday at 18:00.
example_advice = historical_service_advice("Autumn", "Weekday", 18, "No recorded rain")
display(example_advice)

## 14. Service blueprint

| Layer | Before the hour | During the hour | After the hour |
|---|---|---|---|
| Rider | Checks live availability/advisory | Chooses bike or alternative | May give feedback |
| Frontstage service | Shows level, update time, and careful wording | Shows live information if available | Explains changes or service issues |
| Backstage operations | Checks calendar/weather and historical reference | Follows readiness checklist | Records actual outcome and issue |
| Support process | Maintains data and rules | Operations and field coordination | Monthly rule review |
| Evidence | Source, sample size, conditions | Checklist completion and incidents | Demand pattern, complaints, comprehension |

### Current-to-improved process

1. **Current:** demand changes → operations notices late → reactive response.
2. **Improved:** calendar/weather check → historical lookup → readiness level → human review → action and communication → outcome review.

Human review remains important because historical averages cannot observe station availability, unexpected events, or operational disruptions.

## 15. Systems-thinking view

| Feedback loop | Explanation | Management lesson |
|---|---|---|
| Learning loop | Rule → action → outcome → review → improved rule | Record outcomes, not only dashboard views |
| Trust loop | Clear advisory → realistic expectations → trust → continued use | Avoid pretending a citywide level guarantees a bike |
| Reactive-pressure loop | Surprise demand → emergency work → postponed maintenance → future vulnerability | Prepare before recurring pressure windows |

The project demonstrates systems thinking because the analysis is connected to people, process, communication, governance, and feedback—not treated as a charting exercise.

## 16. Pilot plan and KPIs

| Phase | Activity | Evidence needed |
|---|---|---|
| 1. Desk review | Operations staff check whether rules are understandable | Feedback and rule changes |
| 2. Shadow pilot | Generate levels without changing operations | Difference between level and actual demand band |
| 3. Internal pilot | Use the checklist for selected periods | Checklist completion and incident notes |
| 4. Service test | Test rider wording with a small user group | Correct understanding and trust |

| KPI | Simple definition | Why it matters |
|---|---|---|
| High-demand identification | Share of top-quartile actual hours marked High/Critical | Whether the rule notices busy periods |
| False-high rate | High/Critical levels when actual demand is below median | Avoid unnecessary escalation |
| Checklist completion | Completed required actions / required actions | Whether analysis changes operations |
| Rider comprehension | Users who interpret the message correctly / users tested | Prevent misleading communication |
| Service incidents | Recorded issues during each readiness level | Connect the rule to service outcomes |

Only the first two can be approximated from this public dataset. Operational and rider KPIs require a real pilot.

## 17. Executive conclusions and recommendations

1. **Demand pressure is strongly time-dependent.** Recurring weekday and commute-hour patterns justify pre-hour monitoring.
2. **One fixed schedule is not enough.** Season, recorded rain, temperature band, and holiday status change the historical operating context.
3. **Regression adds a controlled association view.** It checks time, calendar, season, and selected weather factors together, without making causal or forecasting claims.
4. **Rules create service value.** Readiness levels connect evidence to operator actions and careful rider messages.
5. **A safe pilot begins internally.** Management should first run the rules in shadow mode, compare them with actual outcomes, and improve them with operational feedback.

### Recommended decision

Run a small **shadow pilot** using the historical lookup and four readiness levels. Do not automate resource allocation and do not publish station-specific claims. Add live station capacity, event, service-incident, and staffing data before making stronger operational decisions.

## 18. What the project can and cannot claim

### Supported

- Describes the observed year of citywide hourly demand.
- Compares historical averages across time, weather, and calendar categories.
- Estimates how selected factors are associated with rentals when considered together.
- Identifies recurring contexts associated with high demand.
- Designs an understandable service-management rule and pilot.

### Not supported

- Future demand prediction, forecasting, or accuracy claims.
- Cause-and-effect conclusions about weather.
- Station-level availability or redistribution.
- Exact bicycle, staffing, or budget requirements.
- Financial savings or return on investment.
- Claims about current Seoul demand from 2017–2018 data.

### Next data required

Live station inventory/capacity, recent multi-year demand, events, trip origins/destinations, failed rental attempts, maintenance actions, staffing/cost data, incidents, and rider feedback.

## 19. Nine-slide project presentation

| Slide | Conclusion-led title | Main evidence | Business message |
|---|---|---|---|
| 1 | Seoul Bike Sharing Demand and Service Planning | Project question | Establish the purpose |
| 2 | Three analytical levels connect evidence to action | Descriptive → regression → planning | Explain the method |
| 3 | The dataset supports a citywide case, not station decisions | Quality and scope audit | Establish credibility |
| 4 | Commute periods create recurring demand pressure | Hourly pattern | Descriptive finding |
| 5 | Season, rain, and calendar type change the context | Grouped comparisons | Descriptive finding |
| 6 | Several factors remain associated when considered together | Regression coefficients | Controlled association, not causation |
| 7 | Historical patterns become four transparent readiness levels | Action matrix | Convert evidence into decisions |
| 8 | Service value depends on process and feedback | Blueprint and systems loops | Analytics is part of a system |
| 9 | A shadow pilot tests usefulness without overclaiming | KPIs, limitations, next data | Learn safely before scaling |

### 45-second opening

> “My project asks how historical demand, weather, and calendar evidence can support a more responsive citywide bike-sharing service. I used three analytical levels. First, descriptive analysis showed what happened. Second, one basic multiple regression examined which factors remained associated when considered together. Third, I translated the evidence into a historical lookup, four readiness levels, and a service pilot. The regression is explanatory, not a forecast or causal model. The project demonstrates how data, people, process, communication, and feedback can be designed as one service system.”

## 20. Interview questions you should be ready to answer

**Why did you exclude non-functioning hours?**  
Because shutdown zeros describe service unavailability, not normal customer demand.

**Why use the mean and median?**  
The mean gives the typical total level but can be influenced by extreme hours; the median gives the middle observation. Showing both makes the lookup more transparent.

**What is a percentile?**  
The 75th percentile is a boundary at or below which 75% of observations fall. I used it only to label the highest quarter as historically high demand.

**Is the lookup a forecast?**  
No. It summarises similar past hours. It is a reference for human planning, not a promise about the future.

**Why use multiple regression?**  
Descriptive comparisons look at conditions separately. Multiple regression estimates the association of each included factor while the others are held constant.

**Why is hour categorical?**  
Demand rises and falls across the day and has separate commute peaks. Treating hour as 0–23 would incorrectly impose one constant straight-line change.

**Why did you remove forecasting and accuracy?**  
The project question is explanatory and managerial: understand historical associations and design transparent actions. Forecasting would require a separate time-based validation and error-analysis objective.

**Can you say rain caused lower demand?**  
No. I can say rainy hours were associated with lower average demand in this dataset. Other conditions may also differ.

**Why is this a service-design project?**  
Because it connects stakeholder needs, data, a decision rule, frontstage communication, backstage operations, governance, KPIs, and feedback.

**What would you improve next?**  
Use current station-level capacity and availability, events, operational actions, costs, and rider feedback; then test the rules in a shadow pilot.